In [41]:
import pandas as pd
import cv2

frames1 = pd.read_csv("kaggle_dataset/labels_train.csv")
frames2 = pd.read_csv("kaggle_dataset/labels_val.csv")
frames3 = pd.read_csv("kaggle_dataset/labels_trainval.csv")

frames = pd.concat([frames1, frames2, frames3])

frames = frames[frames["class_id"] == 1]


def calculate_bounding_box_ratio(row):
    width = row['xmax'] - row['xmin']
    height = row['ymax'] - row['ymin']
    image = cv2.imread("kaggle_dataset/images/" + row['frame'])
    h, w, _ = image.shape
    w_ratio = width / w
    h_ratio = height / h
    ratio = max(w_ratio, h_ratio)
    return ratio

frames["ratio"] = frames.apply(calculate_bounding_box_ratio, axis=1)

frames = frames[frames["ratio"] > 0.2]

frames.drop_duplicates()

print(frames.shape)



(25746, 7)


In [42]:
frames = frames.sort_values(by="ratio", ascending=False)

frames = frames.head(14000)

print(frames.head(10))
print(frames.shape)

                          frame  xmin  xmax  ymin  ymax  class_id     ratio
127927  1479502747761290119.jpg     0   479   273   297         1  0.997917
76780   1479500244091448837.jpg     0   479   264   299         1  0.997917
109479  1479500244091448837.jpg     0   479   264   299         1  0.997917
95228   1479502747761290119.jpg     0   479   273   297         1  0.997917
137894  1479503947844106275.jpg     6   479   268   299         1  0.985417
105195  1479503947844106275.jpg     6   479   268   299         1  0.985417
140662  1479504207359602493.jpg    10   479   275   299         1  0.977083
107963  1479504207359602493.jpg    10   479   275   299         1  0.977083
115428  1479504788401007961.jpg     0   458   268   299         1  0.954167
148127  1479504788401007961.jpg     0   458   268   299         1  0.954167
(14000, 7)


In [43]:
def recalculate_bounding_box(row):
    xmin = row['xmin']
    ymin = row['ymin']
    xmax = row['xmax']
    ymax = row['ymax']

    image = cv2.imread("kaggle_dataset/images/" + row['frame'])
    h, w, _ = image.shape

    new_xmin = int(xmin * (128 / w))
    new_ymin = int(ymin * (128 / h))
    new_xmax = int(xmax * (128 / w))
    new_ymax = int(ymax * (128 / h))

    return new_xmin, new_ymin, new_xmax, new_ymax


for i, row in frames.iterrows():
    new_xmin, new_ymin, new_xmax, new_ymax = recalculate_bounding_box(row)
    frames.at[i, 'xmin'] = new_xmin
    frames.at[i, 'ymin'] = new_ymin
    frames.at[i, 'xmax'] = new_xmax
    frames.at[i, 'ymax'] = new_ymax

print(frames["xmin"].max())
print(frames["xmax"].max())
print(frames["ymin"].max())
print(frames["ymax"].max())


124
127
117
127


In [44]:
import numpy as np

# 70 train 15 val 15 test

merged = (
    frames.groupby("frame")[["xmin", "xmax", "ymin", "ymax"]]
    .apply(lambda x: x.values.tolist())
    .reset_index(name="boxes")
)

frames_train = merged.sample(frac=0.7, random_state=46)
frames_val = merged[~merged.index.isin(frames_train.index)].sample(frac=0.5, random_state=46)
frames_test = merged[~merged.index.isin(pd.concat([frames_train, frames_val]).index)]

np.save("train_frames.npy", frames_train.to_numpy(dtype=object))
np.save("val_frames.npy", frames_val.to_numpy(dtype=object))
np.save("test_frames.npy", frames_test.to_numpy(dtype=object))


In [45]:
import os

os.makedirs("images", exist_ok= True)

for i, row in merged.iterrows():
    image = cv2.imread("kaggle_dataset/images/" + row['frame'], cv2.IMREAD_GRAYSCALE)
    image = cv2.resize(image, (128,128))
    cv2.imwrite("images/" + row['frame'], image)

